<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Two ways to find damage

**[Two ways to find damage](https://learning.nextia-ai.com/courses/vision/m06/two-ways-to-find-damage/)** · Vision AI: See, Count and Inspect · Module 6, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find damaged returns in two ways — a supervised classifier trained on 25 damaged photos, and an anomaly score learned from good photos only — and compare them by AUROC and by Kwame's cost per day.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells and the model (47 MB). No account and no key. |
| **You should know** | Image embeddings, from [Image embeddings](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/); thresholds and costs, from [The return that cost twice](https://learning.nextia-ai.com/courses/vision/m01/the-return-that-cost-twice/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m06/two-ways-to-find-damage/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M06-L01-two-ways-to-find-damage/two-ways-to-find-damage-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 19 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/returns.zip": "dcf02f521f68228d34717b303628b2db8b19d18433126da2ce64047f1449210f",
    "archives/extras.zip": "c2d0ecd8e6fda3846550f27aa054877a0f9daa4f6934d315907ecf107427a104",
    "compare.py": "95b41e2ed829d1e1be658947c266f53d571d9b441c4ed228d2cdec2750f4ed50",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The returns

Each photo shows four tea lights. A damaged photo has at least one damage: a missing piece of wax, a spot, a bent wick, a damaged corner. Damage is rare: 25 of the 405 training photos.

In [ ]:
returns = pd.read_csv("data/returns/manifest.csv")
train, val, test = (returns[returns["split"] == s].reset_index(drop=True) for s in ("train", "val", "test"))
print(returns.groupby(["split", "status"]).size().unstack().to_string())
dam = train[train["status"] == "damaged"].head(4)
show("data/" + dam["file"], list(dam["damage"]), size=3)

## 2. Embeddings of the returns

We use the same ResNet-18 as in the embeddings lesson. It takes about 30 seconds for the 735 photos.

In [ ]:
import torch, timm
from huggingface_hub import hf_hub_download
torch.set_num_threads(4)
# ResNet-18 trained on ImageNet (timm, Apache-2.0 weights), at a fixed revision so that everyone gets the same numbers
weights = hf_hub_download("timm/resnet18.a1_in1k", "model.safetensors", revision="491b427b45c94c7fb0e78b5474cc919aff584bbf")
model = timm.create_model("resnet18.a1_in1k", pretrained=True, num_classes=0, pretrained_cfg_overlay={"file": weights}).eval()
config = timm.data.resolve_data_config({}, model=model)
transform = timm.data.create_transform(**config)

def embed(paths, batch=32):
    """One list of 512 numbers for each photo, scaled to length 1."""
    out = []
    with torch.no_grad():
        for i in range(0, len(paths), batch):
            x = torch.stack([transform(Image.open(p).convert("RGB")) for p in paths[i:i + batch]])
            out.append(model(x).numpy())
    X = np.vstack(out)
    return X / np.linalg.norm(X, axis=1, keepdims=True)

Xtr, Xval, Xte = (embed(list("data/" + df["file"])) for df in (train, val, test))
dtr, dval, dte = ((df["status"] == "damaged").values for df in (train, val, test))
print(Xtr.shape, Xval.shape, Xte.shape)

## 3. Way 1: supervised

Label the damage and train a classifier. We have only 25 damaged training photos, so we give the damaged class the same total weight as the good class (`class_weight="balanced"`). The score is the probability of "damaged".

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced").fit(Xtr, dtr)
sup_val, sup_test = clf.predict_proba(Xval)[:, 1], clf.predict_proba(Xte)[:, 1]
print("AUROC on test:", cp.roc_auc(sup_test, dte))

## 4. Way 2: anomaly-based

Learn only what a **good** candle looks like. The score of a photo is how far it is from its most similar good training photo (1 minus the cosine similarity). No damaged photo is needed to train.

> **Predict.** Which way gives the higher AUROC on the test photos?

In [ ]:
good = Xtr[~dtr]
anom_val, anom_test = 1 - (Xval @ good.T).max(1), 1 - (Xte @ good.T).max(1)
print("AUROC on test:", cp.roc_auc(anom_test, dte))

> **Check.** The anomaly score on whole-photo embeddings has an AUROC of about 0.73: weak. A whole-photo embedding hides a small spot among four candles. PatchCore (the next lesson) compares small **patches** instead, and does far better.

## 5. Kwame's cost per day

AUROC does not pay the bills. `cp.inspection_row` chooses the pass, check and fail lines on the validation photos (at most 100 checks a day for 600 returns) and measures the cost on the test photos.

In [ ]:
rows = [cp.inspection_row("supervised (ResNet-18 + logistic regression)", sup_val, dval, sup_test, dte, list(test["damage"]),
                         "returns test", None, 47, 405, "Apache-2.0"),
        cp.inspection_row("anomaly (nearest good photo)", anom_val, dval, anom_test, dte, list(test["damage"]),
                         "returns test", None, 47, 380, "Apache-2.0")]
for r in rows:
    print(f"{r['method']:45s} cost {r['value']:7.1f} a day | AUROC {r['auroc']} | worst {r['worst']} {r['worst_value']}")
table = cp.Table.load("data/comparison_table.json")
table.show("inspect", columns=("method", "setting", "value", "worst", "worst_value", "labelled_images"))

Compare your two rows with the course's table: anomalib's PatchCore with a wide ResNet-50 costs 368.8 dollars a day on the same test photos, and no inspection costs 2229.4.

## 6. Your turn: the anomaly score's AUROC

> **Your turn.** Put the AUROC of the anomaly score on the **test** photos in `anomaly_auc` (use `cp.roc_auc`).

In [ ]:
anomaly_auc = cp.roc_auc(anom_test, dte)
print(anomaly_auc)

In [ ]:
expect('anomaly_auc', anomaly_auc, 0.73, tolerance=0.03)

## 7. Which way fits Kwame?

Damage at Larkfield is rare and varied: a new kind of damage can appear that no labelled photo shows. A supervised model only knows the damage it saw. An anomaly model flags anything that does not look like a good item, including new damage, and also a new camera (Module 8). Write two sentences for Kwame: which way would you start with, and what must his team label first?

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m06/two-ways-to-find-damage/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Anomaly-based inspection](https://learning.nextia-ai.com/courses/vision/m06/anomaly-based-inspection/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m06/two-ways-to-find-damage/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.